In [27]:
import sqlite3

# Connect to SQLite database (creates 'enterprise_churn.db' in your Colab environment)
conn = sqlite3.connect('enterprise_churn.db')
cursor = conn.cursor()

# Enable foreign key support in SQLite (disabled by default)
cursor.execute("PRAGMA foreign_keys = ON;")

In [28]:
cursor.executescript('''
-- 1. Customers Table
CREATE TABLE IF NOT EXISTS customers (
    customer_id TEXT PRIMARY KEY,
    signup_date TEXT NOT NULL,
    country TEXT NOT NULL,
    subscription_tier TEXT NOT NULL,
    monthly_charges REAL NOT NULL,
    churned INTEGER NOT NULL -- Target variable: 0 (active) or 1 (churned)
);

-- 2. Transactions Table (Linked to Customers)
CREATE TABLE IF NOT EXISTS transactions (
    transaction_id TEXT PRIMARY KEY,
    customer_id TEXT NOT NULL,
    transaction_date TEXT NOT NULL,
    amount REAL NOT NULL,
    payment_status TEXT NOT NULL,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
);

-- 3. Support Tickets Table (Linked to Customers)
CREATE TABLE IF NOT EXISTS support_tickets (
    ticket_id TEXT PRIMARY KEY,
    customer_id TEXT NOT NULL,
    ticket_date TEXT NOT NULL,
    issue_category TEXT NOT NULL,
    resolution_time_hours REAL,
    satisfaction_score INTEGER,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
);

-- 4. Usage Logs Table (Linked to Customers)
CREATE TABLE IF NOT EXISTS usage_logs (
    log_id INTEGER PRIMARY KEY AUTOINCREMENT,
    customer_id TEXT NOT NULL,
    log_date TEXT NOT NULL,
    active_minutes REAL NOT NULL,
    features_used_count INTEGER NOT NULL,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
);
''')

conn.commit()

In [29]:
import pandas as pd

# Example: If you upload 'customers.csv' to your Colab session
# df_customers = pd.read_csv('customers.csv')

# Load DataFrame directly into the SQLite table
# df_customers.to_sql('customers', conn, if_exists='append', index=False)

In [30]:
import sqlite3
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Set random seed for reproducibility
np.random.seed(42)

# 1. Connect to our existing SQLite database
conn = sqlite3.connect('enterprise_churn.db')
cursor = conn.cursor()
cursor.execute("PRAGMA foreign_keys = ON;")

# --- 2. GENERATE CUSTOMERS DATA (500 customers) ---
n_customers = 500
customer_ids = [f"CUST_{i:04d}" for i in range(1, n_customers + 1)]

start_date = datetime.now() - timedelta(days=365)
signup_dates = [start_date + timedelta(days=int(np.random.randint(0, 365))) for _ in range(n_customers)]

countries = np.random.choice(['Egypt', 'USA', 'UAE', 'Germany', 'UK'], size=n_customers, p=[0.4, 0.3, 0.1, 0.1, 0.1])
tiers = np.random.choice(['Basic', 'Pro', 'Enterprise'], size=n_customers, p=[0.5, 0.3, 0.2])

# Base monthly charges depending on tier
tier_charges = {'Basic': 15.0, 'Pro': 45.0, 'Enterprise': 120.0}
monthly_charges = [tier_charges[t] + np.random.normal(0, 2) for t in tiers]

# Churn logic: Enterprise users churn less, Basic users churn more
churn_probs = [0.35 if t == 'Basic' else (0.20 if t == 'Pro' else 0.08) for t in tiers]
churned = [np.random.choice([0, 1], p=[1 - p, p]) for p in churn_probs]

df_customers = pd.DataFrame({
    'customer_id': customer_ids,
    'signup_date': [d.strftime('%Y-%m-%d') for d in signup_dates],
    'country': countries,
    'subscription_tier': tiers,
    'monthly_charges': [round(c, 2) for c in monthly_charges],
    'churned': churned
})

# --- 3. GENERATE TRANSACTIONS DATA ---
transactions = []
for idx, row in df_customers.iterrows():
    c_id = row['customer_id']
    s_date = datetime.strptime(row['signup_date'], '%Y-%m-%d')
    amount = row['monthly_charges']

    # Generate monthly billing records from signup until today
    curr_date = s_date
    while curr_date < datetime.now():
        tx_id = f"TX_{np.random.randint(100000, 999999)}"
        status = np.random.choice(['Success', 'Failed', 'Pending'], p=[0.90, 0.08, 0.02])
        transactions.append({
            'transaction_id': tx_id,
            'customer_id': c_id,
            'transaction_date': curr_date.strftime('%Y-%m-%d'),
            'amount': amount,
            'payment_status': status
        })
        curr_date += timedelta(days=30)

df_transactions = pd.DataFrame(transactions)

# --- 4. GENERATE SUPPORT TICKETS DATA ---
tickets = []
for c_id in customer_ids:
    # Some customers have tickets, some don't
    n_tickets = np.random.choice([0, 1, 2, 3, 4], p=[0.4, 0.3, 0.15, 0.1, 0.05])
    for _ in range(n_tickets):
        t_id = f"TICK_{np.random.randint(10000, 99999)}"
        issue = np.random.choice(['Billing', 'Technical', 'Account', 'Feature Request'], p=[0.3, 0.4, 0.2, 0.1])
        resolution_time = round(np.random.exponential(12), 1) # hours
        satisfaction = int(np.random.choice([1, 2, 3, 4, 5], p=[0.1, 0.1, 0.2, 0.4, 0.2]))

        tickets.append({
            'ticket_id': t_id,
            'customer_id': c_id,
            'ticket_date': (datetime.now() - timedelta(days=int(np.random.randint(1, 300)))).strftime('%Y-%m-%d'),
            'issue_category': issue,
            'resolution_time_hours': resolution_time,
            'satisfaction_score': satisfaction
        })

df_tickets = pd.DataFrame(tickets)

# --- 5. GENERATE USAGE LOGS DATA ---
logs = []
for c_id in customer_ids:
    # 10 log entries per customer representing weekly/monthly check-ins
    for _ in range(10):
        active_mins = max(5.0, round(np.random.normal(120, 40), 1))
        features_count = int(np.random.poisson(5))
        logs.append({
            'customer_id': c_id,
            'log_date': (datetime.now() - timedelta(days=int(np.random.randint(1, 350)))).strftime('%Y-%m-%d'),
            'active_minutes': active_mins,
            'features_used_count': features_count
        })

df_logs = pd.DataFrame(logs)

# --- 6. LOAD DATAFRAMES INTO SQLITE DATABASE ---
df_customers.to_sql('customers', conn, if_exists='replace', index=False)
df_transactions.to_sql('transactions', conn, if_exists='replace', index=False)
df_tickets.to_sql('support_tickets', conn, if_exists='replace', index=False)
df_logs.to_sql('usage_logs', conn, if_exists='replace', index=False)

conn.commit()
conn.close()

print("Database populated successfully with realistic enterprise data!")
print(f"Loaded {len(df_customers)} customers, {len(df_transactions)} transactions, {len(df_tickets)} support tickets, and {len(df_logs)} usage logs.")

Database populated successfully with realistic enterprise data!
Loaded 500 customers, 3230 transactions, 576 support tickets, and 5000 usage logs.


In [31]:
# import sqlite3
# import pandas as pd

# # Re-open the connection to the SQLite database
# conn = sqlite3.connect('enterprise_churn.db')
# cursor = conn.cursor()

# # Define the CTE query to aggregate transactions and usage logs
# query = '''WITH customer_transactions AS (
#     -- Step A: Summarize transactions first (1 row per customer)
#     SELECT
#         customer_id,
#         COUNT(transaction_id) AS total_transactions,
#         SUM(amount) AS total_amount_spent
#     FROM transactions
#     GROUP BY customer_id
# ),
# customer_usage AS (
#     -- Step B: Summarize usage logs first (1 row per customer)
#     SELECT
#         customer_id,
#         AVG(active_minutes) AS avg_active_minutes
#     FROM usage_logs
#     GROUP BY customer_id
# )
# -- Step C: Bring the clean summaries together with the customers table
# SELECT
#     c.customer_id,
#     c.subscription_tier,
#     c.churned,
#     COALESCE(t.total_transactions, 0) AS total_transactions,
#     COALESCE(t.total_amount_spent, 0.0) AS total_amount_spent,
#     COALESCE(u.avg_active_minutes, 0.0) AS avg_active_minutes
# FROM customers c
# LEFT JOIN customer_transactions t ON c.customer_id = t.customer_id
# LEFT JOIN customer_usage u ON c.customer_id = u.customer_id;'''

# # Load the query directly into a pandas DataFrame
# df_summary = pd.read_sql_query(query, conn)

# # Close the connection
# conn.close()

# # Display the top 5 rows of our merged data
# df_summary.head()##

In [32]:
import pandas as pd
import sqlite3

# Connect to your database
conn = sqlite3.connect('enterprise_churn.db')

# Write our professional CTE query as a string
query = """
WITH customer_transactions AS (
    SELECT
        customer_id,
        COUNT(transaction_id) AS total_transactions,
        SUM(amount) AS total_amount_spent
    FROM transactions
    GROUP BY customer_id
),
customer_usage AS (
    SELECT
        customer_id,
        AVG(active_minutes) AS avg_active_minutes
    FROM usage_logs
    GROUP BY customer_id
)
SELECT
    c.customer_id,
    c.subscription_tier,
    c.country,
    c.monthly_charges,
    c.churned,
    COALESCE(t.total_transactions, 0) AS total_transactions,
    COALESCE(t.total_amount_spent, 0.0) AS total_amount_spent,
    COALESCE(u.avg_active_minutes, 0.0) AS avg_active_minutes
FROM customers c
LEFT JOIN customer_transactions t ON c.customer_id = t.customer_id
LEFT JOIN customer_usage u ON c.customer_id = u.customer_id;
"""

# Load query results directly into a Pandas DataFrame
df_model_ready = pd.read_sql_query(query, conn)

# Close connection
conn.close()

# Preview the first 5 rows of our feature dataset
print(df_model_ready.head())

  customer_id subscription_tier country  monthly_charges  churned  \
0   CUST_0001             Basic     USA            11.73        1   
1   CUST_0002             Basic   Egypt            13.85        0   
2   CUST_0003             Basic     USA            16.60        1   
3   CUST_0004               Pro      UK            45.36        0   
4   CUST_0005             Basic     USA            16.07        0   

   total_transactions  total_amount_spent  avg_active_minutes  
0                   9              105.57              107.04  
1                   1               13.85              138.33  
2                   4               66.40               99.48  
3                   9              408.24              100.42  
4                  10              160.70               87.38  


In [33]:
import pandas as pd
import numpy as np
import sqlite3
from datetime import datetime

# Re-initialize df_model_ready if 'signup_date' is missing due to a previous run dropping it
if 'signup_date' not in df_model_ready.columns:
    conn = sqlite3.connect('enterprise_churn.db')
    query = """
    WITH customer_transactions AS (
        SELECT
            customer_id,
            COUNT(transaction_id) AS total_transactions,
            SUM(amount) AS total_amount_spent
        FROM transactions
        GROUP BY customer_id
    ),
    customer_usage AS (
        SELECT
            customer_id,
            AVG(active_minutes) AS avg_active_minutes
        FROM usage_logs
        GROUP BY customer_id
    )
    SELECT
        c.customer_id,
        c.subscription_tier,
        c.country,
        c.monthly_charges,
        c.churned,
        COALESCE(t.total_transactions, 0) AS total_transactions,
        COALESCE(t.total_amount_spent, 0.0) AS total_amount_spent,
        COALESCE(u.avg_active_minutes, 0.0) AS avg_active_minutes
    FROM customers c
    LEFT JOIN customer_transactions t ON c.customer_id = t.customer_id
    LEFT JOIN customer_usage u ON c.customer_id = u.customer_id;
    """
    df_model_ready = pd.read_sql_query(query, conn)
    conn.close()

# Re-merge signup_date from df_customers if it is not present in df_model_ready
if 'signup_date' not in df_model_ready.columns and 'df_customers' in globals():
    df_model_ready = df_model_ready.merge(df_customers[['customer_id', 'signup_date']], on='customer_id', how='left')

# 1. Ensure signup_date is treated as a datetime object
df_model_ready['signup_date'] = pd.to_datetime(df_model_ready['signup_date'])

# 2. FEATURE ENGINEERING: Calculate Customer Tenure (in days from signup until today)
current_date = pd.to_datetime(datetime.now().strftime('%Y-%m-%d'))
df_model_ready['customer_tenure_days'] = (current_date - df_model_ready['signup_date']).dt.days

# 3. FEATURE ENGINEERING: Average Spend Per Transaction
df_model_ready['avg_spend_per_transaction'] = np.where(
    df_model_ready['total_transactions'] > 0,
    df_model_ready['total_amount_spent'] / df_model_ready['total_transactions'],
    0.0
)

# 4. CATEGORICAL ENCODING: Ordinal Encoding for Subscription Tier
tier_mapping = {'Basic': 1, 'Pro': 2, 'Enterprise': 3}
if 'subscription_tier' in df_model_ready.columns:
    df_model_ready['subscription_tier_encoded'] = df_model_ready['subscription_tier'].map(tier_mapping)

# 5. CATEGORICAL ENCODING: One-Hot Encoding for Country (drops original country text column)
if 'country' in df_model_ready.columns:
    df_model_ready = pd.get_dummies(df_model_ready, columns=['country'], drop_first=True, dtype=int)

# 6. Drop redundant raw columns we no longer need for the model
columns_to_drop = [col for col in ['customer_id', 'signup_date', 'subscription_tier'] if col in df_model_ready.columns]
df_model_ready = df_model_ready.drop(columns=columns_to_drop)

# Preview our fully engineered, model-ready feature matrix
print(df_model_ready.head())
print("\nDataset Shape:", df_model_ready.shape)

   monthly_charges  churned  total_transactions  total_amount_spent  \
0            11.73        1                   9              105.57   
1            13.85        0                   1               13.85   
2            16.60        1                   4               66.40   
3            45.36        0                   9              408.24   
4            16.07        0                  10              160.70   

   avg_active_minutes  customer_tenure_days  avg_spend_per_transaction  \
0              107.04                   263                      11.73   
1              138.33                    17                      13.85   
2               99.48                    95                      16.60   
3              100.42                   259                      45.36   
4               87.38                   294                      16.07   

   subscription_tier_encoded  country_Germany  country_UAE  country_UK  \
0                          1                0         

In [34]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# 1. Separate our Features (X) from our Target Variable (y - 'churned')
X = df_model_ready.drop(columns=['churned'])
y = df_model_ready['churned']

# 2. Split data into Training set (80%) and Testing set (20%)
# This ensures our model is tested on data it has NEVER seen before
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Initialize and train our Random Forest model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# 4. Predict churn on our unseen test set
y_pred = model.predict(X_test)

# 5. Evaluate the model's performance
print(f"Model Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%\n")
print("Classification Report:")
print(classification_report(y_test, y_pred))

Model Accuracy: 73.00%

Classification Report:
              precision    recall  f1-score   support

           0       0.77      0.91      0.83        75
           1       0.42      0.20      0.27        25

    accuracy                           0.73       100
   macro avg       0.59      0.55      0.55       100
weighted avg       0.68      0.73      0.69       100



In [35]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# 1. Re-initialize our model with class weighting enabled!
# This tells the model to care equally about both groups.
model_balanced = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42
)

# 2. Train the model again on our training data
model_balanced.fit(X_train, y_train)

# 3. Predict churn on our unseen test set using the fixed model
y_pred_balanced = model_balanced.predict(X_test)

# 4. Print the new performance report card
print("Updated Classification Report (With Class Balancing):")
print(classification_report(y_test, y_pred_balanced))
# 1. Get the raw probability scores from our model instead of final 0/1 guesses
# This gives us the exact percentage chance the model assigns to each customer
probabilities = model_balanced.predict_proba(X_test)

# Let's look at the probability of churning (column index 1) for the first 10 customers in our test set
print("Probability of Churning for the first 10 test customers:")
print(probabilities[:10, 1])
# 1. Set our own custom threshold (30% instead of 50%)
custom_threshold = 0.30

# 2. If the churn probability is >= 0.30, label them as 1 (Churn), otherwise 0 (Stay)
y_pred_custom = (probabilities[:, 1] >= custom_threshold).astype(int)

# 3. Print the new classification report with our custom threshold
from sklearn.metrics import classification_report
print("Classification Report with a 30% Churn Threshold:")
print(classification_report(y_test, y_pred_custom))


Updated Classification Report (With Class Balancing):
              precision    recall  f1-score   support

           0       0.78      0.92      0.84        75
           1       0.45      0.20      0.28        25

    accuracy                           0.74       100
   macro avg       0.61      0.56      0.56       100
weighted avg       0.70      0.74      0.70       100

Probability of Churning for the first 10 test customers:
[0.33 0.38 0.47 0.49 0.01 0.16 0.35 0.07 0.02 0.02]
Classification Report with a 30% Churn Threshold:
              precision    recall  f1-score   support

           0       0.85      0.69      0.76        75
           1       0.41      0.64      0.50        25

    accuracy                           0.68       100
   macro avg       0.63      0.67      0.63       100
weighted avg       0.74      0.68      0.70       100



In [36]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

print(cm)


[[68  7]
 [20  5]]


In [37]:
# Get the probability of each customer belonging to class 1
y_prob = model.predict_proba(X_test)[:, 1]

# Apply a lower classification threshold
y_pred_30 = (y_prob >= 0.30).astype(int)

# Evaluate the new predictions
print(confusion_matrix(y_test, y_pred_30))
print(classification_report(y_test, y_pred_30))
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]

for threshold in thresholds:
    y_pred_threshold = (y_prob >= threshold).astype(int)

    precision = precision_score(y_test, y_pred_threshold, zero_division=0)
    recall = recall_score(y_test, y_pred_threshold, zero_division=0)
    f1 = f1_score(y_test, y_pred_threshold, zero_division=0)

    print(f"Threshold: {threshold}")
    print(f"Precision: {precision:.2f}")
    print(f"Recall: {recall:.2f}")
    print(f"F1-score: {f1:.2f}")
    print("-" * 30)

[[47 28]
 [ 8 17]]
              precision    recall  f1-score   support

           0       0.85      0.63      0.72        75
           1       0.38      0.68      0.49        25

    accuracy                           0.64       100
   macro avg       0.62      0.65      0.60       100
weighted avg       0.74      0.64      0.66       100

Threshold: 0.2
Precision: 0.34
Recall: 0.80
F1-score: 0.48
------------------------------
Threshold: 0.25
Precision: 0.34
Recall: 0.68
F1-score: 0.45
------------------------------
Threshold: 0.3
Precision: 0.38
Recall: 0.68
F1-score: 0.49
------------------------------
Threshold: 0.35
Precision: 0.38
Recall: 0.48
F1-score: 0.42
------------------------------
Threshold: 0.4
Precision: 0.41
Recall: 0.36
F1-score: 0.38
------------------------------
Threshold: 0.45
Precision: 0.47
Recall: 0.32
F1-score: 0.38
------------------------------
Threshold: 0.5
Precision: 0.46
Recall: 0.24
F1-score: 0.32
------------------------------
Threshold: 0.55
Preci

In [38]:
# import pandas as pd

# # 1. Pull the importance scores straight from our trained model
# importances = model_balanced.feature_importances_

# # 2. Create a clean DataFrame pairing each feature name with its score
# feature_importance_df = pd.DataFrame({
#     'Feature': X.columns,
#     'Importance': importances
# }).sort_values(by='Importance', ascending=False)

# # 3. Display the ranked list from most important to least important
# print("Top Churn Drivers (Feature Importance):")
# print(feature_importance_df.to_string(index=False))import pandas as pd

# 1. Extract feature importance scores
importances = model_balanced.feature_importances_

# 2. Match feature names with their importance scores
feature_importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': importances
})

# 3. Sort features from most important to least important
feature_importance_df = feature_importance_df.sort_values(
    by='Importance',
    ascending=False
)

# 4. Display the results
print("Feature Importance Ranking:")
print(feature_importance_df.to_string(index=False))

# 5. Check that importance scores sum to approximately 1
print("\nTotal Importance:", importances.sum())

Feature Importance Ranking:
                  Feature  Importance
       total_amount_spent    0.182619
       avg_active_minutes    0.177428
avg_spend_per_transaction    0.172057
          monthly_charges    0.155629
     customer_tenure_days    0.143923
       total_transactions    0.070320
subscription_tier_encoded    0.041811
               country_UK    0.014895
              country_UAE    0.014838
              country_USA    0.013474
          country_Germany    0.013006

Total Importance: 1.0


In [39]:
from sklearn.inspection import permutation_importance

# Calculate permutation importance
result = permutation_importance(
    model_balanced,
    X_test,
    y_test,
    n_repeats=30,
    random_state=42,
    scoring='f1',
    n_jobs=-1
)

# Create a DataFrame with the results
permutation_importance_df = pd.DataFrame({
    'Feature': X_test.columns,
    'Importance': result.importances_mean,
    'Std': result.importances_std
}).sort_values(by='Importance', ascending=False)

# Display the results
print("Permutation Importance Ranking:")
print(permutation_importance_df.to_string(index=False))

Permutation Importance Ranking:
                  Feature  Importance      Std
       total_amount_spent    0.176301 0.071815
avg_spend_per_transaction    0.135370 0.057209
          monthly_charges    0.134617 0.049311
     customer_tenure_days    0.113762 0.059587
subscription_tier_encoded    0.071590 0.056191
       total_transactions    0.068853 0.047173
               country_UK    0.006479 0.020324
              country_UAE    0.002550 0.015510
          country_Germany   -0.002352 0.012023
       avg_active_minutes   -0.014145 0.066499
              country_USA   -0.023465 0.036256
